# Policy Evaluation Linear Solve

Chapter 15: Bellman Equations and Operators · *Mathematics for AI and Machine Learning* by Fuheng Wu

Contrast iterative policy evaluation with closed-form linear-system solution.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-15/policy-evaluation/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's fixed policy (transition matrix and rewards) and defaults
gamma, K = 0.88, 30
P = torch.tensor([[0.75, 0.25, 0.0],
                  [0.15, 0.75, 0.10],
                  [0.0, 0.25, 0.75]], dtype=torch.float64)
r = torch.tensor([0.2, 0.7, 1.1], dtype=torch.float64)

# ── Closed form: V = r + gamma P V  <=>  (I - gamma P) V = r ────────────────
V_exact = torch.linalg.solve(torch.eye(3, dtype=torch.float64) - gamma * P, r)
print("exact V^pi =", V_exact.tolist())

# ── Iterative evaluation V_{k+1} = r + gamma P V_k from V_0 = 0 ─────────────
V = torch.zeros(3, dtype=torch.float64)
print("\n  k    V_k                              ||V_k - V^pi||_inf   bound gamma^k ||V^pi||")
for k in range(1, K + 1):
    V = r + gamma * P @ V
    if k in (1, 2, 5, 10, 20, K):
        err = (V - V_exact).abs().max().item()
        print(f"{k:3d}   {str([round(v, 4) for v in V.tolist()]):32s} {err:.6f}             {gamma ** k * V_exact.abs().max().item():.6f}")

# V^pi is also the expected discounted return: sum_t gamma^t P^t r (a Neumann series)
series = sum(gamma ** t * torch.linalg.matrix_power(P, t) @ r for t in range(400))
print("\nNeumann series sum gamma^t P^t r =", series.tolist())

# ── Monte Carlo: 4000 rollouts from state 0, all simulated at once ─────────
torch.manual_seed(0)
s = torch.zeros(4000, dtype=torch.long)
G = torch.zeros(4000, dtype=torch.float64)
for t in range(150):
    G += gamma ** t * r[s]
    s = torch.multinomial(P[s], 1).squeeze(1)
print("Monte Carlo return from state 0 =", G.mean().item(), " (exact", V_exact[0].item(), ")")

exact V^pi = [4.066176470588237, 5.375000000000002, 6.713235294117649]

  k    V_k                              ||V_k - V^pi||_inf   bound gamma^k ||V^pi||
  1   [0.2, 0.7, 1.1]                  5.613235             5.907647
  2   [0.486, 1.2852, 1.98]            4.733235             5.198729
  5   [1.4302, 2.6023, 3.7458]         2.967431             3.542789
 10   [2.6207, 3.9128, 5.2262]         1.487005             1.869643
 20   [3.6592, 4.9678, 6.3056]         0.407595             0.520697
 30   [3.9528, 5.2616, 6.5998]         0.113413             0.145015

Neumann series sum gamma^t P^t r = [4.066176470588238, 5.375000000000003, 6.713235294117652]
Monte Carlo return from state 0 = 4.086785355732987  (exact 4.066176470588237 )
